# What Can Experiments Measure? Predictions for Analog Hawking Radiation

**Phase 4 Wave 1 — Item 1A: Experimental Prediction Package (Stakeholder Notebook)**

**Authors:** John Roehm

---

## The bottom line

Black holes emit thermal radiation (Hawking radiation). We cannot measure this from astrophysical black holes — the temperature is far too low. But we **can** build analog black holes in the laboratory using ultracold atomic gases (Bose-Einstein condensates, or BECs), where the speed of sound plays the role of the speed of light.

Our theoretical framework (Schwinger-Keldysh Effective Field Theory, or SK-EFT) predicts specific, testable deviations from the idealized Hawking spectrum. This notebook answers:

1. **What** are the predicted deviations?
2. **How big** are they?
3. **Which experiment** is best positioned to detect each effect?
4. **How many measurements** are needed?

Three experimental groups are actively pursuing these measurements:

| Group | Location | Atom | Status |
|-------|----------|------|--------|
| Steinhauer | Technion, Israel | ⁸⁷Rb | First observation (2019) |
| Heidelberg group | Germany | ³⁹K | Analog cosmology (2024) |
| Trento group | Italy | ²³Na | Spin-sonic proposal |

## Setup

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.core.visualizations import (
    COLORS,
    LAYOUT_TEMPLATE,
    apply_layout,
)
from src.experimental.predictions import (
    compute_prediction_table,
    compute_all_predictions,
    compute_detector_requirements,
    compare_platforms,
    kappa_scaling_prediction,
)
from src.wkb.spectrum import (
    steinhauer_platform,
    heidelberg_platform,
    trento_platform,
    compute_spectrum,
)

platforms = {
    "Steinhauer": steinhauer_platform(),
    "Heidelberg": heidelberg_platform(),
    "Trento": trento_platform(),
}

PLATFORM_COLORS = {
    "Steinhauer": COLORS["Steinhauer"],
    "Heidelberg": COLORS["Heidelberg"],
    "Trento": COLORS["Trento"],
}

## 1. The Three Predicted Effects

Our theory predicts three effects that modify the idealized Hawking spectrum:

1. **Dispersive correction** — The finite healing length of the condensate blue-shifts the effective temperature. This is a small, constant offset.

2. **Dissipative correction** — Phonon damping (analogous to viscosity) heats the spectrum slightly. This grows with frequency.

3. **Quantum noise floor** — The fluctuation-dissipation relation guarantees a minimum occupation number even at very high frequencies. Instead of the thermal spectrum falling to zero, it levels off at a small but nonzero value.

The noise floor is the most distinctive prediction — it has no analog in the non-dissipative Hawking calculation and is a direct consequence of quantum mechanics applied to dissipative systems.

## 2. How Big Are the Effects?

In [ ]:
# viz-ref: fig_prediction_table_comparison

tables = {name: compute_prediction_table(p) for name, p in platforms.items()}

fig = go.Figure()

for name, table in tables.items():
    omegas = [p.omega_over_T_H for p in table.predictions]
    deviations = [p.fractional_deviation * 100 for p in table.predictions]

    fig.add_trace(
        go.Scatter(
            x=omegas,
            y=deviations,
            mode="lines+markers",
            name=name,
            line=dict(color=PLATFORM_COLORS[name], width=2),
            marker=dict(size=8),
        )
    )

fig.add_hline(y=0, line_dash="dot", line_color="grey", opacity=0.5)

apply_layout(
    fig,
    title="How Much Does the Spectrum Deviate from Ideal Hawking?",
    xaxis_title="Frequency (units of Hawking temperature)",
    yaxis_title="Deviation from ideal (%)",
)
fig.show()

The deviations are small (typically below 1%) but grow at higher frequencies. This is because the corrections come from microscopic BEC physics (healing length, phonon damping) which becomes more important when the phonon wavelength approaches the healing length.

The key question for experimentalists: **can current or near-future detectors resolve these differences?**

## 3. How Many Measurements Are Needed?

Detection requires accumulating enough statistics to distinguish the predicted deviation from random noise. The table below shows how many experimental shots (individual BEC realizations) are needed for a statistically significant ($3\sigma$) detection of each effect.

In [ ]:
# viz-ref: fig_detector_requirements

all_reqs = {}
for name, platform in platforms.items():
    all_reqs[name] = compute_detector_requirements(platform)

fig = go.Figure()

goal_labels = ["Dissipative\ncorrection", "Noise\nfloor", "Exact vs\nperturbative"]

for name, reqs in all_reqs.items():
    shots = [req.required_shots for req in reqs]
    fig.add_trace(
        go.Bar(
            x=goal_labels,
            y=shots,
            name=name,
            marker_color=PLATFORM_COLORS[name],
        )
    )

fig.update_layout(barmode="group", yaxis_type="log")
fig.add_hline(
    y=7000,
    line_dash="dash",
    line_color="grey",
    annotation_text="Current best (Steinhauer 2019)",
    annotation_position="top right",
)

apply_layout(
    fig,
    title="How Many Experimental Runs Are Needed?",
    xaxis_title="What to detect",
    yaxis_title="Required shots (log scale)",
)
fig.show()

The dashed line shows the current state of the art (7,000 shots from Steinhauer's 2019 experiment). All three corrections require shot counts far beyond current capabilities. The dissipative correction ($\delta_\mathrm{diss} \sim 10^{-5}$) needs $\sim 10^{12}$ shots — approximately **nine orders of magnitude** beyond current experiments.

These are fundamental predictions of dissipative EFT, but their direct detection requires experimental advances well beyond the current generation. The most accessible test is the **$\kappa$-scaling test** (Section 4): by varying the surface gravity, one can test the *functional form* of the Hawking spectrum without needing to resolve the absolute magnitude of these tiny corrections.

## 4. The $\kappa$-Scaling Test: A Clean Discriminant

The Heidelberg group can tune the atom-atom interaction strength using magnetic fields (Feshbach resonance). This changes the "surface gravity" $\kappa$ of the acoustic black hole.

Our prediction: if you measure the effective temperature at several values of $\kappa$ and plot the deviation, you will see:
- The **dispersive** correction grows as $\kappa^2$ (quadratic)
- The **dissipative** correction stays roughly constant

This different scaling behavior provides a smoking-gun test to identify which type of correction is responsible for any observed deviation.

In [ ]:
# viz-ref: fig_kappa_scaling_phase4

kappa_pred = kappa_scaling_prediction(n_points=20)
D_values = np.linspace(0.01, 0.05, 20)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Dispersive (grows with \u03ba)", "Dissipative (stays flat)"],
)

fig.add_trace(
    go.Scatter(
        x=D_values,
        y=np.abs(kappa_pred.delta_disp_values),
        mode="lines+markers",
        name="Dispersive",
        line=dict(color=COLORS["Steinhauer"], width=2),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=D_values,
        y=np.abs(kappa_pred.delta_diss_values),
        mode="lines+markers",
        name="Dissipative",
        line=dict(color=COLORS["Heidelberg"], width=2),
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="Surface gravity parameter D", row=1, col=1)
fig.update_xaxes(title_text="Surface gravity parameter D", row=1, col=2)
fig.update_yaxes(title_text="Correction size", type="log", row=1, col=1)
fig.update_yaxes(title_text="Correction size", type="log", row=1, col=2)

apply_layout(fig, title="The \u03ba-Scaling Test: How to Tell the Effects Apart")
fig.show()

## 5. Which Experiment Should Do What?

Each experimental platform has different strengths. The table below summarizes which experiment is best positioned for each measurement.

In [ ]:
comparisons = compare_platforms(tables)

rows = []
for comp in comparisons:
    best_key = [k for k, v in comp.rankings.items() if v == 1][0]
    best_name = best_key.split("_")[0]
    rows.append(
        {
            "Measurement": comp.observable,
            "Best platform": best_name,
            "Why": comp.recommendation[:100],
        }
    )

display(pd.DataFrame(rows))

,Measurement,Best platform,Why
0,Maximum spectral deviation from Planckian,Steinhauer,Steinhauer has the largest deviation (1.4e+01)...
1,FDR noise floor visibility (n_noise/n_Planck a...,Steinhauer,Steinhauer has the most visible noise floor (1...
2,UV spectral range (omega_max / T_H),Trento,"Trento has the widest UV range (108.2 T_H), be..."
3,Measurement feasibility (shots for 3-sigma det...,Steinhauer,"Steinhauer requires the fewest shots (27), mak..."


## Summary for Experimentalists

| Effect | Size | Best Platform | Shots Needed | Feasibility |
|--------|------|---------------|--------------|-------------|
| Dissipative correction | $\sim 10^{-5}$ | Steinhauer | $\sim 10^{12}$ | Beyond current technology |
| Noise floor | $\sim 10^{-4} \times n_\mathrm{Planck}$ | Platform-dependent | $\sim 10^{12}$ | Beyond current technology |
| Exact vs perturbative | ~1% at high $\omega$ | Heidelberg (wide UV range) | $\sim 10^{8}$ | Challenging but conceivable |
| $\kappa$-scaling test | Functional form | Heidelberg (Feshbach tuning) | ~5 parameter settings | **Most accessible** |

The **$\kappa$-scaling test** is the most accessible measurement: it requires only a handful of runs at different magnetic field values, and the predicted signature (quadratic vs constant scaling) is qualitatively distinct. It tests the universal structure of the Hawking spectrum without needing to resolve corrections that are many orders of magnitude below current detector sensitivity.

The **noise floor** is the most theoretically distinctive prediction: it has no analog in the non-dissipative Hawking calculation and directly probes the quantum fluctuation-dissipation theorem in curved spacetime.